In [29]:
import pandas as pd


In [30]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("bagavathypriya/spam-ham-dataset")
print("Path to dataset files:", path)
print("\nArchivos disponibles:", os.listdir(path))

for f in os.listdir(path):
    shutil.copy(os.path.join(path, f), f)
    print(f"✅ Copiado: {f}")

Path to dataset files: /root/.cache/kagglehub/datasets/bagavathypriya/spam-ham-dataset/versions/1

Archivos disponibles: ['spamhamdata.csv']
✅ Copiado: spamhamdata.csv


## Dataset preparation
1.   Load the email and keep only the relevant features
2.   Load the Smishing dataset: since it lacks headers, we dynamically assign columns (v1,v2) and rename them for standardization.
3. Merge both datasets to build a more robust
Function execution

In [39]:
def prepare_datasets():
    df_sms = pd.read_csv('spamhamdata.csv', sep='\t', header=None,
                          names=['label', 'original_text'],
                          encoding='latin-1', on_bad_lines='skip')
    df_sms['source'] = 'sms'
    return df_sms

df_base = prepare_datasets()
print(f"Total initial records: {len(df_base)}")
df_base.head()

Total initial records: 5572


,label,original_text,source
0,ham,"Go until jurong point, crazy.. Available only ...",sms
1,ham,Ok lar... Joking wif u oni...,sms
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,sms
3,ham,U dun say so early hor... U c already then say...,sms
4,ham,"Nah I don't think he goes to usf, he lives aro...",sms


In [40]:
def clean_duplicates_nulls(df):
    initial_count = len(df)
    df_clean = df.drop_duplicates(subset=['original_text'], keep='first').copy()
    df_clean = df_clean.dropna(subset=['original_text'])
    final_count = len(df_clean)
    print(f"Removed {initial_count - final_count} duplicate or null records.")
    return df_clean

df_unique = clean_duplicates_nulls(df_base)
print(f"Total records after cleaning: {len(df_unique)}")

Removed 403 duplicate or null records.
Total records after cleaning: 5169


In [41]:
df_unique.to_csv('data_processed_phase1.csv', index=False)
print("Processed data saved to 'data_processed_phase1.csv'\n")
print(df_unique.info())
print("\n--- First 5 records ---")
print(df_unique.head())

Processed data saved to 'data_processed_phase1.csv'

<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   label          5169 non-null   object
 1   original_text  5169 non-null   object
 2   source         5169 non-null   object
dtypes: object(3)
memory usage: 161.5+ KB
None

--- First 5 records ---
  label                                      original_text source
0   ham  Go until jurong point, crazy.. Available only ...    sms
1   ham                      Ok lar... Joking wif u oni...    sms
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...    sms
3   ham  U dun say so early hor... U c already then say...    sms
4   ham  Nah I don't think he goes to usf, he lives aro...    sms


## Feature Engineering (Mathematical Extraction)

In [42]:
def extract_mathematical_features(df):
    df_features = df.copy()
    df_features['message_length'] = df_features['original_text'].apply(lambda x: len(str(x)))
    df_features['num_uppercase'] = df_features['original_text'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
    df_features['num_digits'] = df_features['original_text'].apply(lambda x: sum(1 for c in str(x) if c.isdigit()))
    return df_features

df_final = extract_mathematical_features(df_unique)

## Checkpoint: Post-Feature Extraction Backup

In [43]:
df_final.to_csv('dataset_processed_phase1_pre_nlp.csv', index=False)
print(df_final.info())
print("\n--- First 5 records view ---")
print(df_final.head())

<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   label           5169 non-null   object
 1   original_text   5169 non-null   object
 2   source          5169 non-null   object
 3   message_length  5169 non-null   int64 
 4   num_uppercase   5169 non-null   int64 
 5   num_digits      5169 non-null   int64 
dtypes: int64(3), object(3)
memory usage: 282.7+ KB
None

--- First 5 records view ---
  label                                      original_text source  \
0   ham  Go until jurong point, crazy.. Available only ...    sms   
1   ham                      Ok lar... Joking wif u oni...    sms   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...    sms   
3   ham  U dun say so early hor... U c already then say...    sms   
4   ham  Nah I don't think he goes to usf, he lives aro...    sms   

   message_length  num_uppercase  num_digits 